# Customer Segmentation Case Study

## From customer data to actionable marketing audiences

This case study segments 200 mall customers using annual income and spending score. It covers validation, exploration, feature scaling, model selection, profiling, campaign design, and exportable outputs.

### Business objective

1. Which customers exhibit similar income and spending patterns?
2. What marketing treatment is appropriate for each group?
3. How should the business measure whether those treatments work?

> Segmentation suggests **who** to target. Controlled experiments establish **what creates incremental value**.

## 1. Setup

The notebook downloads a public copy of the Mall Customers dataset. Set `DATA_SOURCE` to a local CSV path if preferred.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

RANDOM_STATE = 42
DEFAULT_DATA_URL = (
    "https://raw.githubusercontent.com/sharmaroshan/"
    "Clustering-of-Mall-Customers/master/Mall_Customers.csv"
)
DATA_SOURCE = os.environ.get("CUSTOMER_DATA_SOURCE", DEFAULT_DATA_URL)
FEATURES = ["Annual Income (k$)", "Spending Score (1-100)"]

## 2. Load and validate the data

In [ ]:
customers = pd.read_csv(DATA_SOURCE).rename(columns={"Genre": "Gender"})
required = {"CustomerID", "Gender", "Age", *FEATURES}
missing = required.difference(customers.columns)

assert not missing, f"Missing columns: {sorted(missing)}"
assert customers["CustomerID"].is_unique, "CustomerID must be unique"
assert customers[list(required)].isna().sum().sum() == 0, "Missing values found"
assert (customers[FEATURES] >= 0).all().all(), "Negative feature values found"

print(f"Rows: {len(customers):,}")
print(f"Columns: {customers.shape[1]}")
display(customers.head())

### Data dictionary

| Variable | Meaning | Modeling role |
|---|---|---|
| CustomerID | Unique identifier | Identifier only |
| Gender | Reported gender | Post-cluster description |
| Age | Customer age | Post-cluster description |
| Annual Income (k$) | Annual income in thousands | Clustering feature |
| Spending Score (1-100) | Retailer-defined score | Clustering feature |

Age and gender describe the groups after modeling; they do not determine campaign eligibility.

## 3. Exploratory analysis

In [ ]:
display(customers[["Age", *FEATURES]].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column, color in zip(
    axes, ["Age", *FEATURES], ["#64748b", "#2563eb", "#7c3aed"]
):
    ax.hist(customers[column], bins=15, color=color, alpha=0.82, edgecolor="white")
    ax.set_title(column)
    ax.set_ylabel("Customers")
fig.suptitle("Customer feature distributions", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for gender, subset in customers.groupby("Gender"):
    ax.scatter(subset[FEATURES[0]], subset[FEATURES[1]],
               alpha=0.7, s=45, label=gender)
ax.set(title="Income and spending before segmentation",
       xlabel=FEATURES[0], ylabel=FEATURES[1])
ax.legend(title="Gender")
plt.tight_layout()
plt.show()

## 4. Prepare the features

K-Means uses Euclidean distance. Standardization places income and spending score on comparable scales.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customers[FEATURES])
display(pd.DataFrame(X_scaled, columns=FEATURES).agg(["mean", "std"]))

## 5. Select the number of clusters

- **Inertia:** look for an elbow where improvement slows.
- **Silhouette score:** higher values indicate better cohesion and separation.
- **Business usefulness:** groups should support distinct actions.

![Model-selection diagnostics](assets/model_selection.svg)

In [ ]:
inertias, silhouette_scores = [], {}
for k in range(1, 11):
    candidate = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = candidate.fit_predict(X_scaled)
    inertias.append(candidate.inertia_)
    if k >= 2:
        silhouette_scores[k] = silhouette_score(X_scaled, labels)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(range(1, 11), inertias, marker="o", color="#2563eb")
axes[0].axvline(5, ls="--", color="#0f766e", label="Selected k=5")
axes[0].set(title="Elbow diagnostic", xlabel="Clusters", ylabel="Inertia")
axes[0].legend()
axes[1].plot(list(silhouette_scores), list(silhouette_scores.values()),
             marker="o", color="#7c3aed")
axes[1].axvline(5, ls="--", color="#0f766e", label="Selected k=5")
axes[1].set(title="Silhouette diagnostic", xlabel="Clusters",
            ylabel="Silhouette score")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Silhouette score for k=5: {silhouette_scores[5]:.3f}")

## 6. Fit the five-segment model

Cluster IDs are arbitrary, so business names are assigned from the unscaled centroid positions.

In [ ]:
model = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=20)
customers["cluster"] = model.fit_predict(X_scaled)

centroids = pd.DataFrame(
    scaler.inverse_transform(model.cluster_centers_), columns=FEATURES
)
centroids["cluster"] = centroids.index

def name_segment(row):
    income, spending = row[FEATURES[0]], row[FEATURES[1]]
    if income < 40:
        return "Promising Spenders" if spending >= 50 else "Budget Conscious"
    if income > 70:
        return "VIP Customers" if spending >= 50 else "Affluent but Unengaged"
    return "Core Customers"

centroids["segment"] = centroids.apply(name_segment, axis=1)
assert centroids["segment"].nunique() == 5
customers["segment"] = customers["cluster"].map(
    centroids.set_index("cluster")["segment"]
)
display(centroids.sort_values(FEATURES[0]).reset_index(drop=True))

## 7. Visualize and profile the segments

![Customer segments](assets/customer_segments.svg)

In [ ]:
SEGMENT_ORDER = [
    "VIP Customers", "Affluent but Unengaged", "Core Customers",
    "Promising Spenders", "Budget Conscious",
]
COLORS = {
    "VIP Customers": "#0f766e", "Affluent but Unengaged": "#d97706",
    "Core Customers": "#2563eb", "Promising Spenders": "#7c3aed",
    "Budget Conscious": "#64748b",
}

fig, ax = plt.subplots(figsize=(10, 6))
for segment in SEGMENT_ORDER:
    subset = customers[customers["segment"] == segment]
    ax.scatter(subset[FEATURES[0]], subset[FEATURES[1]], s=50, alpha=0.76,
               color=COLORS[segment], label=f"{segment} (n={len(subset)})")
ax.scatter(centroids[FEATURES[0]], centroids[FEATURES[1]], marker="X",
           s=220, color="#111827", edgecolor="white", label="Centroids")
ax.set(title="Five actionable customer segments",
       xlabel=FEATURES[0], ylabel=FEATURES[1])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
profiles = (
    customers.groupby("segment")
    .agg(
        customers=("CustomerID", "count"),
        average_age=("Age", "mean"),
        average_income=(FEATURES[0], "mean"),
        average_spending_score=(FEATURES[1], "mean"),
        female_share=("Gender", lambda x: (x == "Female").mean()),
    )
    .reindex(SEGMENT_ORDER)
)
profiles["customer_share"] = profiles["customers"] / len(customers)
display(profiles.style.format({
    "average_age": "{:.1f}", "average_income": "{:.1f}",
    "average_spending_score": "{:.1f}", "female_share": "{:.1%}",
    "customer_share": "{:.1%}",
}))

The chart highlights the activation opportunity among affluent low spenders and the retention value of VIP customers.

![Segment profiles](assets/segment_profiles.svg)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bar_colors = [COLORS[name] for name in profiles.index]
axes[0].barh(profiles.index, profiles["average_income"], color=bar_colors)
axes[0].invert_yaxis()
axes[0].set(title="Average annual income", xlabel="Income (k$)")
axes[1].barh(profiles.index, profiles["average_spending_score"], color=bar_colors)
axes[1].invert_yaxis()
axes[1].set(title="Average spending score", xlabel="Score (1–100)")
plt.tight_layout()
plt.show()

## 8. Marketing strategy by segment

| Segment | Customers | Suggested treatment | Primary KPI |
|---|---:|---|---|
| VIP Customers | 39 | Premium service, exclusives, referrals | Retention / repeat rate |
| Affluent but Unengaged | 35 | Personalized discovery and recommendations | Incremental conversion |
| Core Customers | 81 | Cross-sell, bundles, loyalty milestones | Frequency / order value |
| Promising Spenders | 22 | Points multipliers and accessible bundles | Engagement / repeat rate |
| Budget Conscious | 23 | Value ranges, seasonal offers, price alerts | Redemption / margin |

Treat these actions as hypotheses. Randomize eligible customers within each segment into treatment and control groups and measure incremental lift, profit, and unsubscribes.

## 9. Export reusable outputs

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
customers.to_csv(OUTPUT_DIR / "customer_segment_assignments.csv", index=False)
profiles.reset_index().to_csv(OUTPUT_DIR / "segment_profiles.csv", index=False)
centroids.to_csv(OUTPUT_DIR / "segment_centroids.csv", index=False)

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"Created: {path}")

## 10. Limitations and next steps

- Spending score is a retailer-defined proxy, not transaction revenue or profit.
- The sample is small and cross-sectional; it cannot estimate lifetime value.
- K-Means requires choosing `k` and favors compact distance-based clusters.
- Segment names are interpretations, not ground-truth customer identities.
- Monitor cluster size, centroid movement, and silhouette score over time.
- With transactions, extend the model to RFM, lifetime value, and basket rules.

### Conclusion

The five-cluster solution creates a practical framework: protect VIPs, activate affluent low spenders, grow the core, cultivate promising customers, and serve value seekers efficiently. The next step is randomized testing to quantify incremental value.